# Transform Circuits Data
1. Read bronze circuits table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case ( circuitid → circuit_id, circuitNane → circuit_nane )
4. Rename columns to make them more meaningful ( lat → latitude, long → longitude)
5. Filter out rows where circult_1d is null (business key validation)
6. Remove duplicate records
7. Transform values of columns circuit_name and locality to Title Case
8. Write the transformed data to silver circuits table

### Step 1 - Loading the env config from the common config

In [0]:
%run ../00-common-Config/01-environment-variable

In [0]:
bronze_table=f"{catalog_name}.{bronze_schema}.circuits"
silver_table=f"{catalog_name}.{silver_schema}.circuits"

In [0]:
from pyspark.sql import functions as f

In [0]:
circuits_df=spark.read.table(bronze_table)
display(circuits_df)

### Keep only the columns required for analytics (Drop url column)

In [0]:
circuits_selected_df=circuits_df.select(
    f.col("circuitId"),
    f.col("circuitName"),
    f.col("lat"),
    f.col("long"),
    f.col("locality"),
    f.col("country"),
    f.col("ingestion_timestamp"),
    f.col("source_file")
)

###Step 3 & 4 Standardise column names && rename columns to make them more meaningful
Standardise column names using snake_case ( circuitid → circuit_id, circuitNane → circuit_nane )
Rename columns to make them more meaningful ( lat → latitude, long → longitude)

In [0]:
# circuits_renamed_df=(
#     circuits_selected_df
#     .withColumnRenamed("circuitId","circuit_id")
#     .withColumnRenamed("circuitName","circuit_name")
#     .withColumnRenamed("lat","latitude")
#     .withColumnRenamed("long","longitude")
# )

In [0]:
circuits_renamed_df=(
    circuits_selected_df.withColumnsRenamed({
        "circuitId":"circuit_id",
        "circuitName":"circuit_name",
        "lat":"latitude",
        "long":"longitude"
        })
)

### Step 5 - Filter out rows where circult_1d is null (business key validation)

In [0]:
# circuits_valid_df=circuits_renamed_df.filter("circuit_id is not NULL")
# display(circuits_valid_df)


In [0]:
circuits_valid_df=circuits_renamed_df.filter(
    f.col("circuit_id").isNotNull()
)
display(circuits_valid_df)

### Step 6. Remove duplicate records

In [0]:
circuits_distinct_df=circuits_valid_df.distinct()
display(circuits_distinct_df)
# This removed the duplicate record from the table but we want to removed the duplicate cicuits_id

In [0]:
circuits_dup_df=circuits_valid_df.dropDuplicates(["circuit_id"])
display(circuits_dup_df)
# It will remove duplicate circuit_id value (but it is random)

### Step 7. Transform values of columns circuit_name and locality to Title Case

In [0]:
circuits_final_df=(
    circuits_dup_df
    .withColumn("circuit_name",f.initcap(f.col("circuit_name")))
    .withColumn("locality",f.initcap(f.col("locality")))
                )

In [0]:
circuits_final_df.write.mode("overwrite").format("delta").saveAsTable(silver_table)

In [0]:
spark.table(silver_table).show()